In [2]:
import pyspark
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import countDistinct
spark = SparkSession.builder.appName('practice').getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/07/30 21:19:32 WARN Utils: Your hostname, Janna-MBP.local, resolves to a loopback address: 127.0.0.1; using 10.0.0.26 instead (on interface en0)
25/07/30 21:19:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/30 21:19:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


1. Show first five rows:

In [3]:
df = spark.read.option('header', 'true').csv('week2-ds.tsv', sep = '\t')
df.show(5)

+--------------------+--------------------+-----------------------+-----------+-------+-----------+-----------+---------------+
|          Polygon ID|    Hashed Device ID|Unix Timestamp of Visit|Data Source|   Date|Time of Day|Day of Week|      Time Zone|
+--------------------+--------------------+-----------------------+-----------+-------+-----------+-----------+---------------+
|Blue Nile Blue Ni...|9f0bc33ace6a4d2bd...|             1643151728|          X|1/25/22|   16:02:08|        Tue|America/Phoenix|
|Blue Nile Blue Ni...|ca58bf6ffe8a0cef2...|             1643131230|          X|1/25/22|   10:20:30|        Tue|America/Phoenix|
|Blue Nile Blue Ni...|0c275358edf2e911b...|             1643153377|          X|1/25/22|   16:29:37|        Tue|America/Phoenix|
|Blue Nile Blue Ni...|30a03daca53ba0f54...|             1643142816|          X|1/25/22|   13:33:36|        Tue|America/Phoenix|
|Blue Nile Blue Ni...|b7c057bbd95d16c97...|             1643125135|          X|1/25/22|    8:38:55|     

2. Number of Rows

In [4]:
rows = df.count()
print(rows)

27307


3. Number of Distinct Rows

In [5]:
unique_rows = df.distinct().count()
print(unique_rows)

26307


4. Number of Distinct 'Hashed Device ID'

In [6]:
unique_rows = df.select('Hashed Device ID').distinct().count()
print(unique_rows)

7518


5. Number of Distinct Columns

In [7]:
unique_rows = df.select('Polygon ID').distinct().count()
print(f'Distinct Polygon IDs: {unique_rows}')


unique_rows = df.select('Data Source').distinct().count()
print(f'Distinct Data Sources: {unique_rows}')

unique_rows = df.select('Date').distinct().count()
print(f'Distinct Dates: {unique_rows}')

Distinct Polygon IDs: 7
Distinct Data Sources: 4
Distinct Dates: 182


6a. Date , Number of distinct Hashed Device IDs

In [8]:
df.groupBy('Date').agg(countDistinct('Hashed Device ID')).orderBy("Date").show()

+-------+--------------------------------+
|   Date|count(DISTINCT Hashed Device ID)|
+-------+--------------------------------+
| 1/1/22|                              78|
|1/10/22|                              41|
|1/11/22|                              44|
|1/12/22|                              44|
|1/13/22|                              47|
|1/14/22|                              64|
|1/15/22|                             121|
|1/16/22|                              62|
|1/17/22|                              71|
|1/18/22|                              35|
|1/19/22|                              31|
| 1/2/22|                              61|
|1/20/22|                              42|
|1/21/22|                              48|
|1/22/22|                              99|
|1/23/22|                              54|
|1/24/22|                              31|
|1/25/22|                              29|
|1/26/22|                              39|
|1/27/22|                              36|
+-------+--

6b. Polygon ID, Number of Distinct Hashed Device IDs

In [9]:
df.groupBy('Polygon ID').agg(countDistinct('Hashed Device ID')).show(truncate=False)

+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|Polygon ID                                                                                                                   |count(DISTINCT Hashed Device ID)|
+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|Blue Nile Blue Nile Jewelry - Houston Galleria, Houston, TX|11050619                                                         |1962                            |
|Blue Nile Blue Nile Jewelry - Garden State Plaza, Paramus, NJ|11050629                                                       |726                             |
|Blue Nile Washington Square, Portland, OR|9695391                                                                            |164                             |
|Blue Nile The Mall at Rockingham 

6c. Polygon ID, Date, Number of Distinct Hashed Device IDs


In [10]:
df.groupBy(['Polygon ID', 'Date']).agg(countDistinct('Hashed Device ID')).orderBy("Polygon ID", "Date").show(truncate=False)

+-------------------------------------------------------------------+-------+--------------------------------+
|Polygon ID                                                         |Date   |count(DISTINCT Hashed Device ID)|
+-------------------------------------------------------------------+-------+--------------------------------+
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/1/22 |10                              |
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/10/22|8                               |
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/11/22|9                               |
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/12/22|5                               |
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/13/22|7                               |
|Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/14/22|7                               |
|

7a. Source X


In [11]:
df_x = df.where(df['Data Source'] == 'X')
df_x.groupBy(['Data Source','Date']).agg(countDistinct('Hashed Device ID')).orderBy("Date").show()


+-----------+-------+--------------------------------+
|Data Source|   Date|count(DISTINCT Hashed Device ID)|
+-----------+-------+--------------------------------+
|          X| 1/1/22|                              70|
|          X|1/10/22|                              35|
|          X|1/11/22|                              35|
|          X|1/12/22|                              39|
|          X|1/13/22|                              41|
|          X|1/14/22|                              56|
|          X|1/15/22|                             115|
|          X|1/16/22|                              57|
|          X|1/17/22|                              66|
|          X|1/18/22|                              30|
|          X|1/19/22|                              22|
|          X| 1/2/22|                              56|
|          X|1/20/22|                              38|
|          X|1/21/22|                              40|
|          X|1/22/22|                              82|
|         

In [12]:
df_x.groupBy(['Data Source', 'Polygon ID']).agg(countDistinct('Hashed Device ID')).orderBy("Polygon ID").show(truncate=False)

+-----------+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|Data Source|Polygon ID                                                                                                                   |count(DISTINCT Hashed Device ID)|
+-----------+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|X          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622                                                          |238                             |
|X          |Blue Nile Blue Nile Jewelry - Fashion Square, Scottsdale, AZ|11088822                                                        |2337                            |
|X          |Blue Nile Blue Nile Jewelry - Garden State Plaza, Paramus, NJ|11050629                                                    

In [13]:
df_x.groupBy('Data Source','Polygon ID', 'Date').agg(countDistinct('Hashed Device ID')).orderBy("Polygon ID", "Date").show(truncate=False)

+-----------+-------------------------------------------------------------------+-------+--------------------------------+
|Data Source|Polygon ID                                                         |Date   |count(DISTINCT Hashed Device ID)|
+-----------+-------------------------------------------------------------------+-------+--------------------------------+
|X          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/1/22 |7                               |
|X          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/10/22|7                               |
|X          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/11/22|5                               |
|X          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/12/22|3                               |
|X          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/13/22|4                               |
|X          |Blu

7b. Source C

In [14]:
df_c = df.where(df['Data Source'] == 'C')
df_c.groupBy(['Date', 'Data Source']).agg(countDistinct('Hashed Device ID')).orderBy("Date").show()

+-------+-----------+--------------------------------+
|   Date|Data Source|count(DISTINCT Hashed Device ID)|
+-------+-----------+--------------------------------+
| 1/1/22|          C|                               7|
|1/10/22|          C|                               7|
|1/11/22|          C|                              10|
|1/12/22|          C|                               5|
|1/13/22|          C|                               4|
|1/14/22|          C|                               8|
|1/15/22|          C|                               6|
|1/16/22|          C|                               4|
|1/17/22|          C|                               4|
|1/18/22|          C|                               4|
|1/19/22|          C|                               8|
| 1/2/22|          C|                               4|
|1/20/22|          C|                               2|
|1/21/22|          C|                               6|
|1/22/22|          C|                              24|
|1/23/22| 

In [15]:
df_c.groupBy(['Data Source', 'Polygon ID']).agg(countDistinct('Hashed Device ID')).orderBy("Polygon ID").show(truncate=False)

+-----------+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|Data Source|Polygon ID                                                                                                                   |count(DISTINCT Hashed Device ID)|
+-----------+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|C          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622                                                          |86                              |
|C          |Blue Nile Blue Nile Jewelry - Fashion Square, Scottsdale, AZ|11088822                                                        |101                             |
|C          |Blue Nile Blue Nile Jewelry - Garden State Plaza, Paramus, NJ|11050629                                                    

In [16]:
df_c.groupBy('Data Source','Polygon ID', 'Date').agg(countDistinct('Hashed Device ID')).orderBy("Polygon ID", "Date").show(truncate=False)

+-----------+-------------------------------------------------------------------+-------+--------------------------------+
|Data Source|Polygon ID                                                         |Date   |count(DISTINCT Hashed Device ID)|
+-----------+-------------------------------------------------------------------+-------+--------------------------------+
|C          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/1/22 |3                               |
|C          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/10/22|2                               |
|C          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/11/22|5                               |
|C          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/12/22|2                               |
|C          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/13/22|2                               |
|C          |Blu

7c. Source Y

In [17]:
df_y = df.where(df['Data Source'] == 'Y')
df_y.groupBy([ 'Data Source','Date']).agg(countDistinct('Hashed Device ID')).orderBy("Date").show()

+-----------+-------+--------------------------------+
|Data Source|   Date|count(DISTINCT Hashed Device ID)|
+-----------+-------+--------------------------------+
|          Y| 1/1/22|                               3|
|          Y|1/11/22|                               7|
|          Y|1/12/22|                               3|
|          Y|1/13/22|                               3|
|          Y|1/14/22|                               3|
|          Y|1/15/22|                               6|
|          Y|1/16/22|                               3|
|          Y|1/17/22|                               4|
|          Y|1/18/22|                               4|
|          Y|1/19/22|                               8|
|          Y| 1/2/22|                               2|
|          Y|1/20/22|                               4|
|          Y|1/21/22|                               6|
|          Y|1/22/22|                              11|
|          Y|1/23/22|                               7|
|         

In [18]:
df_y.groupBy(['Data Source', 'Polygon ID']).agg(countDistinct('Hashed Device ID')).orderBy("Polygon ID").show(truncate=False)

+-----------+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|Data Source|Polygon ID                                                                                                                   |count(DISTINCT Hashed Device ID)|
+-----------+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|Y          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622                                                          |58                              |
|Y          |Blue Nile Blue Nile Jewelry - Fashion Square, Scottsdale, AZ|11088822                                                        |112                             |
|Y          |Blue Nile Blue Nile Jewelry - Garden State Plaza, Paramus, NJ|11050629                                                    

In [19]:
df_y.groupBy('Data Source','Polygon ID', 'Date').agg(countDistinct('Hashed Device ID')).orderBy("Polygon ID", "Date").show(truncate=False)

+-----------+-------------------------------------------------------------------+-------+--------------------------------+
|Data Source|Polygon ID                                                         |Date   |count(DISTINCT Hashed Device ID)|
+-----------+-------------------------------------------------------------------+-------+--------------------------------+
|Y          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/1/22 |1                               |
|Y          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/11/22|5                               |
|Y          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/12/22|2                               |
|Y          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/13/22|2                               |
|Y          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/14/22|2                               |
|Y          |Blu

7d. Source Z

In [20]:
df_z = df.where(df['Data Source'] == 'Z')
df_z.groupBy([ 'Data Source','Date']).agg(countDistinct('Hashed Device ID')).orderBy("Date").show()

+-----------+-------+--------------------------------+
|Data Source|   Date|count(DISTINCT Hashed Device ID)|
+-----------+-------+--------------------------------+
|          Z|1/11/22|                               3|
|          Z|1/14/22|                               2|
|          Z|1/15/22|                               1|
|          Z|1/19/22|                               2|
|          Z|1/23/22|                               3|
|          Z|1/24/22|                               1|
|          Z|1/25/22|                               2|
|          Z|1/26/22|                               2|
|          Z|1/27/22|                               2|
|          Z|1/28/22|                               1|
|          Z| 1/3/22|                               2|
|          Z|1/31/22|                               2|
|          Z| 1/4/22|                               1|
|          Z| 1/6/22|                               1|
|          Z| 1/7/22|                               3|
|         

In [21]:
df_z.groupBy(['Data Source', 'Polygon ID']).agg(countDistinct('Hashed Device ID')).orderBy("Polygon ID").show(truncate=False)

+-----------+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|Data Source|Polygon ID                                                                                                                   |count(DISTINCT Hashed Device ID)|
+-----------+-----------------------------------------------------------------------------------------------------------------------------+--------------------------------+
|Z          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622                                                          |81                              |
|Z          |Blue Nile Blue Nile Jewelry - Fashion Square, Scottsdale, AZ|11088822                                                        |453                             |
|Z          |Blue Nile Blue Nile Jewelry - Garden State Plaza, Paramus, NJ|11050629                                                    

In [22]:
df_z.groupBy('Data Source','Polygon ID', 'Date').agg(countDistinct('Hashed Device ID')).orderBy("Polygon ID", "Date").show(truncate=False)

+-----------+-------------------------------------------------------------------+-------+--------------------------------+
|Data Source|Polygon ID                                                         |Date   |count(DISTINCT Hashed Device ID)|
+-----------+-------------------------------------------------------------------+-------+--------------------------------+
|Z          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/11/22|1                               |
|Z          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/15/22|1                               |
|Z          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/19/22|1                               |
|Z          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/23/22|1                               |
|Z          |Blue Nile Blue Nile Jewelry - Domain Northside, Austin, TX|11050622|1/24/22|1                               |
|Z          |Blu

In [23]:
unique_zone = df.select('Time Zone').distinct().count()
print(f'Unique Time Zones: {unique_zone}')

Unique Time Zones: 4


In [24]:
spark.stop()